# 7.11 DenseNet & EfficientNet

DenseNet reuses features by concatenation, while EfficientNet scales depth, width, and resolution together under a compute budget.

Dense blocks make each layer see all previous features, so channel counts grow predictably. EfficientNet treats model scaling as a compound choice instead of increasing just one dimension. The same arithmetic exposes memory costs, compression, and budget surprises.

Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)


## The concept, built once (D1)
$$C_L=C_0+kL,\qquad \mathrm{cost}\approx d\,w^2\,r^2$$

We first write the reusable method and assert the exact lesson numbers before scaling to the ladder.

In [ ]:

def dense_or_compound_block(c0=16, growth=12, layers=4):
    channels = [c0 + growth * i for i in range(1, layers + 1)]
    layer_weights = 3 * 3 * 40 * growth
    compressed = int(channels[-1] * 0.5)
    depth = 1.2 ** 2
    width = 1.1 ** 2
    resolution = 1.15 ** 2
    compute_factor = depth * 1.4641 * 1.74800625
    return channels, layer_weights, compressed, depth, width, resolution, compute_factor


channels, layer_weights, compressed, depth, width, resolution, compute_factor = dense_or_compound_block()

assert channels[0] == 28
assert channels[1] == 40
assert channels[3] == 64
assert layer_weights == 4320
assert compressed == 32
assert np.isclose(np.round(compute_factor, 3), 3.685)

print("dense channels", channels)
print("layer weights", layer_weights)
print("compressed channels", compressed)
print("compound factors", np.round([depth, width, resolution, compute_factor], 4))


## Visual check
The numbers above are easier to trust when the intermediate feature behavior is visible.

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(8, 3))

axes[0].plot([1, 2, 3, 4], channels, marker="o")
axes[0].set_title("DenseNet channel growth")
axes[0].set_xlabel("layer")
axes[0].set_ylabel("channels")

axes[1].bar(["depth", "width", "resolution", "compute"], [depth, width, resolution, compute_factor])
axes[1].set_title("EfficientNet phi=2")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()


## Dataset ladder (D1 to D5)
We inline the shared CPU-safe classification ladder. Each rung returns images `X` with shape `(n, 8, 8)` and labels `y`, so the same featurizer can be evaluated from hand patches to the hardest fallback or cached MNIST rung.

In [ ]:
"""
F6 (Vision) shared dataset ladder — D1..D5 of rising complexity, CPU-only and run-all-safe.

This is the canonical ladder inlined into the classification-style Part-7 notebooks. Every
rung returns (X, y) with X shape (n, 8, 8) float in [0, 1] and integer labels y, so one
featurizer + classifier can run unchanged across all five rungs (the "watch it scale" story).

D4/D5 load real MNIST / CIFAR-10 via torchvision when the download is available (as in Colab),
offline they fall back to a harder synthetic set so run-all never fails. Code is written one
statement per line for readability.
"""

import numpy as np
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split


def _resize_to_8x8(img):
    """Nearest-neighbour resize of a 2-D array to 8x8 (no SciPy dependency)."""
    h, w = img.shape
    rows = (np.linspace(0, h - 1, 8)).round().astype(int)
    cols = (np.linspace(0, w - 1, 8)).round().astype(int)
    return img[np.ix_(rows, cols)]


def _normalize(x):
    """Scale an array into [0, 1], a flat array becomes all zeros."""
    x = x.astype(float)
    lo = x.min()
    hi = x.max()
    if hi - lo < 1e-12:
        return np.zeros_like(x)
    return (x - lo) / (hi - lo)


def d1_hand_patches():
    """D1 — hand-built 4x4 patches, 2 classes: a vertical line (col 1) vs a horizontal line (row 1).

    Fixed positions with light jitter, so the two classes are cleanly separable and the
    mechanism is fully visible — the easy first rung.
    """
    rng = np.random.default_rng(0)
    images = []
    labels = []
    for _ in range(24):
        patch = rng.uniform(0.0, 0.15, size=(4, 4))
        patch[:, 1] = rng.uniform(0.85, 1.0)
        images.append(_resize_to_8x8(patch))
        labels.append(0)
    for _ in range(24):
        patch = rng.uniform(0.0, 0.15, size=(4, 4))
        patch[1, :] = rng.uniform(0.85, 1.0)
        images.append(_resize_to_8x8(patch))
        labels.append(1)
    return np.array(images), np.array(labels)


def d2_synthetic_shapes():
    """D2 — clean synthetic shapes on an 8x8 grid, 2 classes (square vs disc)."""
    rng = np.random.default_rng(1)
    yy, xx = np.mgrid[0:8, 0:8]
    images = []
    labels = []
    for _ in range(80):
        img = rng.uniform(0.0, 0.1, size=(8, 8))
        img[2:6, 2:6] = 0.9
        images.append(img)
        labels.append(0)
    for _ in range(80):
        img = rng.uniform(0.0, 0.1, size=(8, 8))
        disc = (xx - 3.5) ** 2 + (yy - 3.5) ** 2 <= 4.0
        img[disc] = 0.9
        images.append(img)
        labels.append(1)
    return np.array(images), np.array(labels)


def d3_sklearn_digits():
    """D3 — real sklearn digits (native 8x8), 4 classes for a fast, honest multi-class rung."""
    digits = load_digits()
    keep = np.isin(digits.target, [0, 1, 2, 3])
    X = digits.images[keep]
    y = digits.target[keep]
    X = np.array([_normalize(img) for img in X])
    return X, y


def _synthetic_textured(n_per_class, n_classes, noise, seed):
    """A harder synthetic fallback: textured class prototypes at 8x8 with noise."""
    rng = np.random.default_rng(seed)
    protos = [rng.uniform(0.0, 1.0, size=(8, 8)) for _ in range(n_classes)]
    images = []
    labels = []
    for cls in range(n_classes):
        for _ in range(n_per_class):
            img = protos[cls] + rng.normal(0.0, noise, size=(8, 8))
            images.append(_normalize(img))
            labels.append(cls)
    return np.array(images), np.array(labels)


def _call_with_timeout(fn, seconds):
    """Run fn() but abort with TimeoutError after `seconds` (guards slow/hanging downloads)."""
    import signal

    def _raise(signum, frame):
        raise TimeoutError("download timed out")

    old = signal.signal(signal.SIGALRM, _raise)
    signal.alarm(seconds)
    try:
        return fn()
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old)


def _load_mnist_gray(classes, n_per_class, seed, shift=False, noise=0.0):
    """Load MNIST via torchvision, grayscale + resize to 8x8, subsample. Raises on failure.

    MNIST is a small (~11 MB) real dataset. CIFAR-10 is deliberately avoided (a 170 MB
    download breaks run-all-safety), the harder D5 rung instead shifts and noises MNIST.
    """
    import torchvision

    ds = torchvision.datasets.MNIST(root="./data", train=True, download=True)
    rng = np.random.default_rng(seed)
    targets = np.array(ds.targets)
    images = []
    labels = []
    for cls in classes:
        idx = np.where(targets == cls)[0][:n_per_class]
        for i in idx:
            arr = np.asarray(ds[int(i)][0], dtype=float)
            small = _resize_to_8x8(arr)
            if shift:
                small = np.roll(small, rng.integers(-1, 2), axis=0)
                small = np.roll(small, rng.integers(-1, 2), axis=1)
            if noise:
                small = small + rng.normal(0.0, noise * 255.0, size=(8, 8))
            images.append(_normalize(small))
            labels.append(cls)
    return np.array(images), np.array(labels)


def d4_mnist_or_fallback():
    """D4 — real MNIST (4 clean classes) when downloadable, else a harder synthetic set."""
    try:
        X, y = _call_with_timeout(lambda: _load_mnist_gray([0, 1, 2, 3], 60, seed=4), 30)
        return (X, y), "MNIST (real)"
    except Exception:
        return _synthetic_textured(60, 4, noise=0.35, seed=4), "synthetic (offline fallback)"


def d5_mnist_hard_or_fallback():
    """D5 — real MNIST, more classes with shift + noise (distribution shift), else hardest synthetic."""
    try:
        X, y = _call_with_timeout(lambda: _load_mnist_gray([0, 1, 2, 3, 4, 5], 60, seed=5, shift=True, noise=0.12), 30)
        return (X, y), "MNIST shifted+noisy (real, harder)"
    except Exception:
        return _synthetic_textured(60, 6, noise=0.6, seed=5), "synthetic (offline fallback)"


def load_ladder():
    """Return the five rungs as a list of (name, X, y). D4/D5 note whether real data loaded."""
    rungs = []
    rungs.append(("D1 hand patches", *d1_hand_patches()))
    rungs.append(("D2 synthetic shapes", *d2_synthetic_shapes()))
    rungs.append(("D3 sklearn digits", *d3_sklearn_digits()))
    (x4, y4), tag4 = d4_mnist_or_fallback()
    rungs.append((f"D4 {tag4}", x4, y4))
    (x5, y5), tag5 = d5_mnist_hard_or_fallback()
    rungs.append((f"D5 {tag5}", x5, y5))
    return rungs


def accuracy_with(featurize, X, y):
    """Map each image through featurize, train logistic regression, return held-out accuracy."""
    feats = np.array([featurize(img) for img in X])
    x_tr, x_te, y_tr, y_te = train_test_split(feats, y, test_size=0.4, random_state=0, stratify=y)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_tr, y_tr)
    return clf.score(x_te, y_te)




rungs = load_ladder()

fig, axes = plt.subplots(1, 5, figsize=(12, 3))

for ax, (name, X, y) in zip(axes, rungs):
    ax.imshow(X[0], cmap="gray")
    ax.set_title(f"{name.split()[0]}\n{X.shape}\n{len(set(y.tolist()))} classes")
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

plt.tight_layout()
plt.show()

for name, X, y in rungs:
    print(f"{name:38s} X={X.shape} classes={sorted(set(y.tolist()))}")


## Run the same method across D1-D5
Only the data rung changes. The featurizer and accuracy metric stay fixed.

In [ ]:

def topic_feature_map(img):
    reuse_1 = img
    reuse_2 = np.maximum(img - img.mean(), 0.0)
    reuse_3 = np.abs(np.diff(img, axis=0, append=img[-1:, :]))
    dense_average = (reuse_1 + reuse_2 + reuse_3) / 3.0
    return dense_average


def featurize(img):
    dense_average = topic_feature_map(img)
    compressed_stats = np.array([dense_average.mean(), dense_average.std(), dense_average.max()])
    return np.concatenate([img.ravel(), dense_average.ravel(), compressed_stats])


accuracies = []

for name, X, y in rungs:
    acc = accuracy_with(featurize, X, y)
    accuracies.append(acc)
    print(f"{name:38s} accuracy={acc:.3f}")

baseline_accuracies = []

for name, X, y in rungs:
    acc = accuracy_with(lambda im: im.ravel(), X, y)
    baseline_accuracies.append(acc)

print("flat baseline", [round(x, 3) for x in baseline_accuracies])


## Results visualization
Top row: one feature or activation panel per rung. Bottom row: accuracy versus ladder complexity.

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(14, 6))

for idx, (name, X, y) in enumerate(rungs):
    feature = topic_feature_map(X[0])
    if isinstance(feature, tuple):
        feature = feature[0]
    axes[0, idx].imshow(feature, cmap="viridis")
    axes[0, idx].set_title(name.split()[0])
    axes[0, idx].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

axes[1, 0].plot(range(1, 6), accuracies, marker="o", label="topic features")
axes[1, 0].plot(range(1, 6), baseline_accuracies, marker="s", label="flat baseline")
axes[1, 0].set_xticks(range(1, 6))
axes[1, 0].set_xlabel("rung")
axes[1, 0].set_ylabel("accuracy")
axes[1, 0].set_ylim(0.0, 1.05)
axes[1, 0].legend()
axes[1, 0].set_title("accuracy vs rung")

for ax in axes[1, 1:]:
    ax.axis("off")

plt.tight_layout()
plt.show()


## Pitfall on D5: confusing concat with add
DenseNet concatenates features, so channels grow instead of staying fixed like a residual add. We show the channel blow-up and the compute undercount, then apply compression and the full compound budget formula.

In [ ]:

add_channels = 16
concat_channels = channels[-1]
wrong_budget = depth * width * resolution
fixed_budget = compute_factor
capped_channels = compressed

print("ResNet-style add would keep", add_channels, "channels")
print("DenseNet concat grows to", concat_channels, "channels")
print("underestimated budget", np.round(wrong_budget, 3))
print("compound budget", np.round(fixed_budget, 3))
print("after compression", capped_channels, "channels")


## Evaluate it + Practice
- Metric: held-out accuracy on every rung, compared with a no-skill flat-pixel logistic baseline.
- Sanity check: D1 should be easy enough to overfit or nearly overfit with the concept features.
- Ablation: remove the key block feature and accuracy should not improve over the flat baseline.
- Failure signal: the hardest rung may expose distribution shift, shape mismatch, or compute-budget mistakes before D1 does.

Practice prompts:
1. Change one design constant and rerun the accuracy curve.
2. Print the D5 confusion pattern for the worst two classes.
3. Replace the feature map panel with an example from a different class.

In [ ]:
# Your code here


In [ ]:
# Your code here


In [ ]:
# Your code here
